# RQ3 — Model Interpretability using SHAP

This notebook explains **what drives the XGBoost predictions produced in RQ2**.

**Design principle:** the sample filter, feature set, target transformation, XGBoost
hyperparameters and random seed are *identical* to `Modelling_3.ipynb` (RQ2).
Cell 7 verifies this numerically by reproducing the RQ2 out-of-fold MAE.
If that check fails, RQ2 and RQ3 are describing different models and the
chapter is not defensible.

**Two methodological points to address in the write-up before running anything:**

1. **`Source` is a data-provenance variable, not a project characteristic.** If SHAP
   ranks it highly, that is an artefact of how the dataset was assembled (different
   sources record different populations of projects), not a managerial finding.
   Cell 16 runs a sensitivity analysis without `Source` so you can state whether the
   substantive drivers survive its removal.
2. **The feature set is small (one numeric, two categorical).** SHAP cannot reveal
   drivers that were never in the model. Frame RQ3 as *"what does the model use, and
   is that consistent with theory?"* — not *"what causes cost overruns?"*

## 1. Environment

In [ ]:
# Colab: install SHAP (restart not required)
!pip install shap --quiet

In [ ]:
# ============================================================
# RQ3: Interpretability of the XGBoost cost overrun model
# ============================================================
#
# Research question:
#   Which project characteristics drive the gradient boosted
#   model's cost overrun predictions, and are those drivers
#   consistent with the megaproject literature?
#
# Method:
#   - SHAP (Lundberg and Lee, 2017), TreeExplainer
#   - Attributions computed OUT-OF-FOLD, mirroring the RQ2
#     validation design (10-fold CV x 20 repeats)
#   - One-hot contributions aggregated back to the original
#     categorical variables before interpretation
#
# Attribution scale:
#   SHAP values are additive on the model's output scale, which
#   here is ln(1 + Cost_Overrun_Pct / 100). They are NOT in
#   percentage points. Cell 9 provides a multiplicative
#   translation for the write-up.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shap

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("shap version:", shap.__version__)

In [ ]:
# Colab only: upload the dataset if it is not already in the session.
# Skip this cell if you are mounting Google Drive instead.

from google.colab import files

if not os.path.exists("Master_Dataset_Rebased_2025_v6.xlsx"):
    print("Upload Master_Dataset_Rebased_2025_v6.xlsx")
    files.upload()
else:
    print("Dataset already present in session.")

## 2. Configuration — must match RQ2 exactly

In [ ]:
# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------
# Every value in this block is copied from RQ2. Do not change
# one without changing the other.

INPUT_XLSX  = "Master_Dataset_Rebased_2025_v6.xlsx"
OUTPUT_XLSX = "RQ3_SHAP_Results.xlsx"
FIG_DIR     = "RQ3_Figures"
SHEET       = "Model_Ready"

N_SPLITS     = 10
N_REPEATS    = 20
RANDOM_STATE = 42
EXPECTED_N   = 107

# RQ2 XGBoost specification, reproduced verbatim
XGB_PARAMS = dict(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=5.0,
    random_state=RANDOM_STATE,
    n_jobs=1,
    objective="reg:squarederror",
)

# RQ2 benchmark to reproduce (mean out-of-fold MAE, percentage points)
RQ2_XGB_MAE = 26.838
MAE_TOLERANCE = 0.05

os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

def save_fig(fig, filename):
    fig.savefig(os.path.join(FIG_DIR, filename + ".png"))
    fig.savefig(os.path.join(FIG_DIR, filename + ".svg"))
    plt.close(fig)

## 3. Data

In [ ]:
# ------------------------------------------------------------
# 3. Load and filter data  (identical to RQ2)
# ------------------------------------------------------------

df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET)

df = (
    df[df["Meets_500M_2025"] == "Yes"]
    .copy()
    .reset_index(drop=True)
)

if len(df) != EXPECTED_N:
    raise ValueError(
        f"Expected {EXPECTED_N} projects after filtering, but found {len(df)}."
    )

print(f"Analytical sample: N = {len(df)} projects")

REQUIRED_COLUMNS = [
    "Cost_Overrun_Pct",
    "Orig_USD_PPP_2025_M",
    "Sector_Std",
    "Source",
]

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nMissing values:")
print(df[REQUIRED_COLUMNS].isna().sum())

In [ ]:
# ------------------------------------------------------------
# 4. Target and predictors  (identical to RQ2)
# ------------------------------------------------------------

df["overrun_pct"] = pd.to_numeric(df["Cost_Overrun_Pct"], errors="raise")
df["orig_size_m"] = pd.to_numeric(df["Orig_USD_PPP_2025_M"], errors="raise")

if (df["orig_size_m"] <= 0).any():
    raise ValueError("Orig_USD_PPP_2025_M contains zero or negative values.")

if (df["overrun_pct"] <= -100).any():
    raise ValueError("Cost_Overrun_Pct contains a value <= -100%.")

df["log_size"] = np.log(df["orig_size_m"])
df["y_log"]    = np.log1p(df["overrun_pct"] / 100.0)

NUM = ["log_size"]
CAT = ["Sector_Std", "Source"]
FEATURES = NUM + CAT

X     = df[FEATURES].copy()
y_pct = df["overrun_pct"].to_numpy(dtype=float)
y_log = df["y_log"].to_numpy(dtype=float)
n     = len(df)

id_candidates = ["Project_ID", "Project", "Project_Name", "Name"]
ID_COL = next((c for c in id_candidates if c in df.columns), None)

if ID_COL is not None:
    project_ids = df[ID_COL].astype(str).to_numpy()
else:
    project_ids = np.array([f"Project_{i+1:03d}" for i in range(n)])

print(f"Predictors: {FEATURES}")
print(f"Target: ln(1 + Cost_Overrun_Pct / 100)")
print(f"Projects: {n}")
print(f"\nSector_Std levels: {df['Sector_Std'].nunique()}")
print(df["Sector_Std"].value_counts().to_string())
print(f"\nSource levels: {df['Source'].nunique()}")
print(df["Source"].value_counts().to_string())

## 5. Encoding

RQ2 fitted the one-hot encoder inside each training fold. Here the encoder is fitted
once on the full predictor matrix instead, so that the SHAP matrix has a **stable
column set across all 200 fits** and attributions can be averaged.

This is legitimate and should be stated in the methodology: one-hot encoding estimates
no parameters from the outcome and applies no scaling statistics, so fitting it on the
full predictor matrix leaks no target information. Cell 7 confirms the resulting model
is numerically identical to RQ2.

In [ ]:
# ------------------------------------------------------------
# 5. Encode once, so SHAP columns are stable across folds
# ------------------------------------------------------------

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:                       # older scikit-learn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUM),
        ("cat", ohe, CAT),
    ]
)

X_enc_array = preprocess.fit_transform(X)
X_enc_array = np.asarray(X_enc_array, dtype=float)

raw_names = list(preprocess.get_feature_names_out())


def map_to_variable(name):
    # Returns (original_variable, level_or_None)
    clean = name.split("__", 1)[1] if "__" in name else name
    for c in CAT:
        if clean.startswith(c + "_"):
            return c, clean[len(c) + 1:]
    return clean, None


feature_names = []
feature_variable = []
feature_level = []

for nm in raw_names:
    var, lvl = map_to_variable(nm)
    feature_variable.append(var)
    feature_level.append(lvl)
    feature_names.append(f"{var} = {lvl}" if lvl is not None else var)

X_enc = pd.DataFrame(X_enc_array, columns=feature_names)
n_enc = X_enc.shape[1]

# Display labels for the three original variables
VARIABLES = ["log_size"] + CAT
VAR_LABELS = {
    "log_size":   "Project size (log USD PPP 2025)",
    "Sector_Std": "Sector",
    "Source":     "Data source",
}

var_columns = {
    v: [i for i, fv in enumerate(feature_variable) if fv == v]
    for v in VARIABLES
}

print(f"Encoded design matrix: {X_enc.shape[0]} rows x {n_enc} columns\n")
for v in VARIABLES:
    print(f"{VAR_LABELS[v]:35s} -> {len(var_columns[v]):2d} column(s)")

## 6. Out-of-fold SHAP attribution

In [ ]:
# ------------------------------------------------------------
# 6. Repeated out-of-fold SHAP
# ------------------------------------------------------------
# For every fold: fit on the training rows, then explain only
# the held-out rows. Each project therefore receives an
# attribution from a model that never saw it, matching the
# out-of-fold logic used for RQ2 predictive performance.

shap_oof  = np.full((N_REPEATS, n, n_enc), np.nan)
base_vals = np.full((N_REPEATS, n), np.nan)
pred_oof  = np.full((N_REPEATS, n), np.nan)

print(f"Computing SHAP: {N_SPLITS}-fold CV x {N_REPEATS} repeats "
      f"= {N_SPLITS * N_REPEATS} model fits")

for rep in range(N_REPEATS):

    kf = KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE + rep,
    )

    for train_idx, test_idx in kf.split(X_enc):

        model = XGBRegressor(**XGB_PARAMS)
        model.fit(X_enc.iloc[train_idx], y_log[train_idx])

        # Out-of-fold prediction, back-transformed to pp
        pred_log = model.predict(X_enc.iloc[test_idx])
        pred_oof[rep, test_idx] = (np.exp(pred_log) - 1.0) * 100.0

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_enc.iloc[test_idx])

        shap_oof[rep, test_idx, :] = np.asarray(sv)
        base_vals[rep, test_idx] = float(np.ravel(explainer.expected_value)[0])

    if (rep + 1) % 5 == 0:
        print(f"  completed repeat {rep + 1} of {N_REPEATS}")

if np.isnan(shap_oof).any():
    raise RuntimeError("Incomplete SHAP coverage — some projects were never held out.")

print("\nOut-of-fold SHAP computation complete.")
print(f"SHAP array shape: {shap_oof.shape}  (repeats x projects x features)")

In [ ]:
# ------------------------------------------------------------
# 7. VERIFICATION — reproduce the RQ2 XGBoost result
# ------------------------------------------------------------
# If this fails, the model being explained is not the model
# reported in RQ2 and the chapter is not internally consistent.

mae_per_repeat = np.array([
    mean_absolute_error(y_pct, pred_oof[rep]) for rep in range(N_REPEATS)
])

mae_here = mae_per_repeat.mean()

print(f"RQ3 mean out-of-fold MAE : {mae_here:.3f} pp")
print(f"RQ2 reported MAE         : {RQ2_XGB_MAE:.3f} pp")
print(f"Difference               : {abs(mae_here - RQ2_XGB_MAE):.4f} pp")

if abs(mae_here - RQ2_XGB_MAE) > MAE_TOLERANCE:
    raise AssertionError(
        "RQ3 model does not reproduce the RQ2 XGBoost result. "
        "Check hyperparameters, seed, sample filter and feature set."
    )

print("\nPASS — the model explained here is the model reported in RQ2.")

# Additivity check: base value + sum of SHAP = model output (log scale)
recon = base_vals[0] + shap_oof[0].sum(axis=1)
actual_log_pred = np.log1p(pred_oof[0] / 100.0)
max_dev = np.max(np.abs(recon - actual_log_pred))

print(f"Max additivity deviation : {max_dev:.2e} (log units)")
print("SHAP values sum exactly to the model output, as required.")

## 8. Aggregating one-hot contributions back to variables

In [ ]:
# ------------------------------------------------------------
# 8. Variable-level SHAP
# ------------------------------------------------------------
# A categorical variable is spread across several one-hot
# columns. Because SHAP is additive, the contribution of the
# variable for a given project is the SUM of its columns.
# Summing must happen BEFORE taking absolute values, otherwise
# offsetting dummy contributions are double counted.

shap_var = np.zeros((N_REPEATS, n, len(VARIABLES)))

for vi, v in enumerate(VARIABLES):
    cols = var_columns[v]
    shap_var[:, :, vi] = shap_oof[:, :, cols].sum(axis=2)

# Mean absolute contribution, computed per repeat then summarised
imp_per_repeat = np.abs(shap_var).mean(axis=1)          # (repeats, variables)

importance = pd.DataFrame({
    "Variable":      [VAR_LABELS[v] for v in VARIABLES],
    "Mean_abs_SHAP": imp_per_repeat.mean(axis=0),
    "SD_across_repeats": imp_per_repeat.std(axis=0, ddof=1),
})

importance["Share_of_total"] = (
    importance["Mean_abs_SHAP"] / importance["Mean_abs_SHAP"].sum()
)

# Translation to a reportable scale.
# SHAP values are in ln(1 + overrun/100) units. A contribution s
# multiplies the predicted cost ratio by exp(s), i.e. shifts the
# predicted overrun by (exp(s) - 1) x 100 percent of the ratio.
importance["Approx_multiplicative_effect_%"] = (
    np.exp(importance["Mean_abs_SHAP"]) - 1.0
) * 100.0

importance = importance.sort_values(
    "Mean_abs_SHAP", ascending=False
).reset_index(drop=True)

print("Global feature importance (out-of-fold SHAP, averaged over repeats)\n")
print(importance.round(4).to_string(index=False))

In [ ]:
# ------------------------------------------------------------
# 9. Encoded-feature importance (individual categories)
# ------------------------------------------------------------

imp_enc_per_repeat = np.abs(shap_oof).mean(axis=1)      # (repeats, encoded features)

importance_enc = pd.DataFrame({
    "Feature":       feature_names,
    "Variable":      [VAR_LABELS[v] for v in feature_variable],
    "Level":         feature_level,
    "Mean_abs_SHAP": imp_enc_per_repeat.mean(axis=0),
    "SD_across_repeats": imp_enc_per_repeat.std(axis=0, ddof=1),
})

# Signed mean effect among projects that actually belong to the level
signed = []
n_projects_at_level = []

shap_mean = shap_oof.mean(axis=0)                       # (projects, encoded features)

for j, lvl in enumerate(feature_level):
    if lvl is None:
        signed.append(shap_mean[:, j].mean())
        n_projects_at_level.append(n)
    else:
        mask = X_enc.iloc[:, j].to_numpy() == 1
        signed.append(shap_mean[mask, j].mean() if mask.any() else np.nan)
        n_projects_at_level.append(int(mask.sum()))

importance_enc["Mean_signed_SHAP_when_present"] = signed
importance_enc["N_projects"] = n_projects_at_level

importance_enc = importance_enc.sort_values(
    "Mean_abs_SHAP", ascending=False
).reset_index(drop=True)

print("Encoded feature importance\n")
print(importance_enc.round(4).to_string(index=False))

## 10. Figures

In [ ]:
# ------------------------------------------------------------
# Figure 1: Global importance by variable, with stability bars
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 4.5))

order = importance.iloc[::-1]

ax.barh(
    order["Variable"],
    order["Mean_abs_SHAP"],
    xerr=order["SD_across_repeats"],
    capsize=4,
)

ax.set_xlabel("Mean |SHAP| contribution (log cost-ratio units)")
ax.set_title(
    "What the model uses to predict cost overrun",
    fontweight="bold",
)

for i, (val, share) in enumerate(
    zip(order["Mean_abs_SHAP"], order["Share_of_total"])
):
    ax.text(val + 0.004, i, f"{share:.0%}", va="center")

ax.set_xlim(0, order["Mean_abs_SHAP"].max() * 1.25)
fig.tight_layout()
save_fig(fig, "Fig9_shap_global_importance")
plt.close("all")

print("Saved Fig9_shap_global_importance")
print("Error bars show variation across the 20 cross-validation repeats.")

In [ ]:
# ------------------------------------------------------------
# Figure 2: Beeswarm — direction and spread of contributions
# ------------------------------------------------------------
# Uses the mean out-of-fold SHAP value per project.

explanation = shap.Explanation(
    values=shap_mean,
    base_values=base_vals.mean(axis=0),
    data=X_enc.to_numpy(),
    feature_names=feature_names,
)

plt.figure()
shap.plots.beeswarm(explanation, max_display=12, show=False)
plt.title("Distribution of SHAP contributions across projects", fontweight="bold")
fig = plt.gcf()
fig.tight_layout()
save_fig(fig, "Fig10_shap_beeswarm")
plt.close("all")

print("Saved Fig10_shap_beeswarm")

In [ ]:
# ------------------------------------------------------------
# Figure 3: Dependence — does project size push predictions up
#           or down, and is the relationship monotonic?
# ------------------------------------------------------------

size_idx = VARIABLES.index("log_size")
size_shap = shap_var[:, :, size_idx].mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(
    df["orig_size_m"],
    size_shap,
    c=y_pct,
    alpha=0.75,
)

ax.set_xscale("log")
ax.axhline(0, linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Original project cost (USD PPP 2025, millions, log scale)")
ax.set_ylabel("SHAP contribution of project size\n(log cost-ratio units)")
ax.set_title(
    "Effect of project size on the predicted overrun",
    fontweight="bold",
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Actual cost overrun (%)")

fig.tight_layout()
save_fig(fig, "Fig11_shap_dependence_size")
plt.close("all")

# Direction of the size effect, for the write-up
median_size = df["orig_size_m"].median()
small = size_shap[df["orig_size_m"] <= median_size].mean()
large = size_shap[df["orig_size_m"] > median_size].mean()

print("Saved Fig11_shap_dependence_size\n")
print(f"Median project size: {median_size:,.0f}m USD PPP 2025")
print(f"Mean SHAP, below-median projects: {small:+.4f}")
print(f"Mean SHAP, above-median projects: {large:+.4f}")
print(
    "\nInterpretation: a positive value means the model pushes the predicted "
    "overrun upward for projects of that size."
)

In [ ]:
# ------------------------------------------------------------
# Figure 4: Which sectors and sources move predictions, and how
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, v in zip(axes, CAT):

    sub = (
        importance_enc[importance_enc["Variable"] == VAR_LABELS[v]]
        .dropna(subset=["Mean_signed_SHAP_when_present"])
        .sort_values("Mean_signed_SHAP_when_present")
    )

    colors = [
        "tab:red" if x > 0 else "tab:blue"
        for x in sub["Mean_signed_SHAP_when_present"]
    ]

    ax.barh(sub["Level"], sub["Mean_signed_SHAP_when_present"], color=colors)
    ax.axvline(0, linestyle="--", linewidth=1, color="black")
    ax.set_xlabel("Mean SHAP contribution when present\n(log cost-ratio units)")
    ax.set_title(VAR_LABELS[v], fontweight="bold")

    ax.margins(x=0.18)

    for i, (val, nproj) in enumerate(
        zip(sub["Mean_signed_SHAP_when_present"], sub["N_projects"])
    ):
        ax.annotate(
            f"n={nproj}",
            xy=(val, i),
            xytext=(5 if val >= 0 else -5, 0),
            textcoords="offset points",
            va="center",
            ha="left" if val >= 0 else "right",
            fontsize=9,
        )

fig.suptitle(
    "Red raises the predicted overrun, blue lowers it",
    fontweight="bold",
)
fig.tight_layout()
save_fig(fig, "Fig12_shap_category_effects")
plt.close("all")

print("Saved Fig12_shap_category_effects")
print("\nRead the n values carefully. A large effect resting on two or three "
      "projects is not a finding.")

## 11. Local explanations

Global importance answers *what the model uses on average*. A distinction-level chapter
should also show *how the model reasons about an individual project*, because that is
the form in which a practitioner would actually encounter the tool at appraisal.

In [ ]:
# ------------------------------------------------------------
# Local explanations for three illustrative projects
# ------------------------------------------------------------

mean_pred = pred_oof.mean(axis=0)
abs_err   = np.abs(mean_pred - y_pct)

# Candidate cases, in priority order. Duplicates are dropped so the
# same project is not presented twice under two labels.
candidates = [
    ("Largest actual overrun",    np.argsort(-y_pct)),
    ("Project closest to median", np.argsort(np.abs(y_pct - np.median(y_pct)))),
    ("Largest prediction error",  np.argsort(-abs_err)),
]

cases, used = {}, set()
for label, order in candidates:
    for idx in order:
        if int(idx) not in used:
            cases[label] = int(idx)
            used.add(int(idx))
            break

shap_var_mean = shap_var.mean(axis=0)          # (projects, variables)
base_mean     = base_vals.mean(axis=0)

# For local plots, show the untransformed project size: it is more
# readable than the logged value and the attribution is unchanged.
LOCAL_LABELS = ["Project size (USD PPP 2025, m)", "Sector", "Data source"]

local_rows = []

for label, idx in cases.items():

    exp_local = shap.Explanation(
        values=shap_var_mean[idx],
        base_values=base_mean[idx],
        data=np.array([
            round(float(df["orig_size_m"].iloc[idx])),
            df["Sector_Std"].iloc[idx],
            df["Source"].iloc[idx],
        ], dtype=object),
        feature_names=LOCAL_LABELS,
    )

    plt.figure()
    shap.plots.waterfall(exp_local, show=False)
    plt.title(
        f"{label}\n{project_ids[idx]} — actual {y_pct[idx]:.0f}%, "
        f"predicted {mean_pred[idx]:.0f}%",
        fontweight="bold",
        pad=22,
    )
    fig = plt.gcf()
    fig.tight_layout()
    save_fig(fig, f"Fig13_local_{label.replace(' ', '_').lower()}")
    plt.close("all")

    row = {
        "Case": label,
        "Project": project_ids[idx],
        "Size_USD_M": df["orig_size_m"].iloc[idx],
        "Sector": df["Sector_Std"].iloc[idx],
        "Source": df["Source"].iloc[idx],
        "Actual_overrun_pct": y_pct[idx],
        "Predicted_overrun_pct": mean_pred[idx],
    }
    for vi, v in enumerate(VARIABLES):
        row[f"SHAP_{v}"] = shap_var_mean[idx, vi]

    local_rows.append(row)

local_cases = pd.DataFrame(local_rows)

print(local_cases.round(3).to_string(index=False))
print("\nSaved three local explanation figures.")

## 12. Triangulation against the linear baseline

In [ ]:
# ------------------------------------------------------------
# 12. Cross-check: do the linear coefficients tell the same story?
# ------------------------------------------------------------
# Convergence between a black-box attribution method and a
# transparent parametric model is a credibility argument. If SHAP
# and the regression coefficients disagree in sign, say so and
# investigate rather than reporting only the convenient one.
#
# drop="first" is used here so the coefficients are identified
# against an explicit reference category.

try:
    ohe_ref = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:
    ohe_ref = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)

pre_ref = ColumnTransformer(
    transformers=[("num", "passthrough", NUM), ("cat", ohe_ref, CAT)]
)

X_ref = pre_ref.fit_transform(X)
ref_names = [
    (map_to_variable(nm)[0] + " = " + map_to_variable(nm)[1])
    if map_to_variable(nm)[1] is not None else map_to_variable(nm)[0]
    for nm in pre_ref.get_feature_names_out()
]

lr = LinearRegression().fit(X_ref, y_log)

lr_coefs = pd.DataFrame({
    "Term": ref_names,
    "Coefficient_log_units": lr.coef_,
})
lr_coefs["Multiplicative_effect_%"] = (np.exp(lr.coef_) - 1.0) * 100.0

reference_levels = {
    c: sorted(df[c].dropna().unique())[0] for c in CAT
}

print("Reference categories (omitted, absorbed into the intercept):")
for k, v in reference_levels.items():
    print(f"  {k}: {v}")

print(f"\nIntercept: {lr.intercept_:.4f}\n")
print(lr_coefs.round(4).to_string(index=False))

print(
    "\nCompare the sign of the log_size coefficient with the size dependence "
    "plot in Figure 11. Agreement strengthens the interpretive claim; "
    "disagreement is itself a finding worth reporting."
)

## 13. Sensitivity analysis — removing the data-source variable

This is the cell that protects the chapter. `Source` records *where the observation came
from*, not a property of the project. If it carries substantial SHAP weight, the model
is partly learning which dataset a project was drawn from — a form of dataset artefact
rather than a cost-overrun mechanism.

Report both specifications. If the size and sector attributions are stable across them,
say so explicitly; if they are not, the honest conclusion is that the interpretation is
conditional on data provenance.

In [ ]:
# ------------------------------------------------------------
# 13. Sensitivity: refit and re-explain without Source
# ------------------------------------------------------------

CAT_S = ["Sector_Std"]
FEATURES_S = NUM + CAT_S
X_s = df[FEATURES_S].copy()

try:
    ohe_s = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_s = OneHotEncoder(handle_unknown="ignore", sparse=False)

pre_s = ColumnTransformer(
    transformers=[("num", "passthrough", NUM), ("cat", ohe_s, CAT_S)]
)

X_s_enc = np.asarray(pre_s.fit_transform(X_s), dtype=float)

s_names, s_variable = [], []
for nm in pre_s.get_feature_names_out():
    var, lvl = map_to_variable(nm)
    s_variable.append(var)
    s_names.append(f"{var} = {lvl}" if lvl is not None else var)

X_s_enc = pd.DataFrame(X_s_enc, columns=s_names)

VARIABLES_S = ["log_size", "Sector_Std"]
s_cols = {
    v: [i for i, fv in enumerate(s_variable) if fv == v] for v in VARIABLES_S
}

shap_s = np.full((N_REPEATS, n, X_s_enc.shape[1]), np.nan)
pred_s = np.full((N_REPEATS, n), np.nan)

for rep in range(N_REPEATS):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE + rep)
    for train_idx, test_idx in kf.split(X_s_enc):
        m = XGBRegressor(**XGB_PARAMS)
        m.fit(X_s_enc.iloc[train_idx], y_log[train_idx])
        pred_s[rep, test_idx] = (np.exp(m.predict(X_s_enc.iloc[test_idx])) - 1.0) * 100.0
        shap_s[rep, test_idx, :] = np.asarray(
            shap.TreeExplainer(m).shap_values(X_s_enc.iloc[test_idx])
        )

shap_var_s = np.zeros((N_REPEATS, n, len(VARIABLES_S)))
for vi, v in enumerate(VARIABLES_S):
    shap_var_s[:, :, vi] = shap_s[:, :, s_cols[v]].sum(axis=2)

imp_s = np.abs(shap_var_s).mean(axis=1)

mae_s = np.mean([mean_absolute_error(y_pct, pred_s[r]) for r in range(N_REPEATS)])

sensitivity = pd.DataFrame({
    "Variable": [VAR_LABELS[v] for v in VARIABLES_S],
    "Mean_abs_SHAP_with_Source": [
        importance.loc[importance["Variable"] == VAR_LABELS[v], "Mean_abs_SHAP"].iloc[0]
        for v in VARIABLES_S
    ],
    "Mean_abs_SHAP_without_Source": imp_s.mean(axis=0),
})

sensitivity["Change_%"] = (
    (sensitivity["Mean_abs_SHAP_without_Source"]
     / sensitivity["Mean_abs_SHAP_with_Source"] - 1.0) * 100.0
)

print("SENSITIVITY ANALYSIS: attribution stability when Source is removed\n")
print(sensitivity.round(4).to_string(index=False))

print(f"\nOut-of-fold MAE with Source   : {mae_here:.3f} pp")
print(f"Out-of-fold MAE without Source: {mae_s:.3f} pp")
print(f"Change in predictive accuracy : {mae_s - mae_here:+.3f} pp")

source_share = importance.loc[
    importance["Variable"] == VAR_LABELS["Source"], "Share_of_total"
].iloc[0]

print(f"\nSource accounts for {source_share:.0%} of total attributed importance.")
if source_share > 0.30:
    print(
        "This is high. The chapter must argue explicitly that the model is not "
        "simply learning dataset provenance, or concede the limitation."
    )

## 14. Export

In [ ]:
# ------------------------------------------------------------
# 14. Excel workbook
# ------------------------------------------------------------

BOLD   = Font(name="Arial", size=10, bold=True)
ARIAL  = Font(name="Arial", size=10)
HEADER = PatternFill("solid", fgColor="D9D9D9")
CENTER = Alignment(horizontal="center", vertical="center")
THIN   = Side(style="thin", color="BFBFBF")
BOX    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def write_sheet(wb, title, frame, widths=None, first=False):
    ws = wb.active if first else wb.create_sheet(title)
    if first:
        ws.title = title
    ws.append(list(frame.columns))
    for _, row in frame.iterrows():
        ws.append([
            v.item() if hasattr(v, "item") else v for v in row.tolist()
        ])
    for c in range(1, len(frame.columns) + 1):
        cell = ws.cell(1, c)
        cell.font = BOLD
        cell.fill = HEADER
        cell.alignment = CENTER
        cell.border = BOX
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = ARIAL
            cell.border = BOX
            if isinstance(cell.value, float):
                cell.number_format = "0.0000"
    ws.freeze_panes = "A2"
    for i, c in enumerate(frame.columns, start=1):
        ws.column_dimensions[get_column_letter(i)].width = (
            widths.get(c, 22) if widths else max(14, min(38, len(str(c)) + 6))
        )
    return ws


# Project-level variable attributions
project_shap = pd.DataFrame({
    "Project_ID": project_ids,
    "Size_USD_PPP_2025_M": df["orig_size_m"].to_numpy(),
    "Sector_Std": df["Sector_Std"].to_numpy(),
    "Source": df["Source"].to_numpy(),
    "Actual_overrun_pct": y_pct,
    "Predicted_overrun_pct": mean_pred,
})
for vi, v in enumerate(VARIABLES):
    project_shap[f"SHAP_{v}"] = shap_var_mean[:, vi]
project_shap["SHAP_base_value"] = base_mean

config_rows = pd.DataFrame({
    "Parameter": [
        "Analysis",
        "Research question",
        "Analytical sample",
        "Predictors",
        "Target transformation",
        "Attribution method",
        "Attribution scale",
        "Validation design",
        "Model fits explained",
        "Encoder fitting",
        "Aggregation rule",
        "RQ2 reproduction check",
        "Sensitivity analysis",
        "Triangulation",
        "Random seed",
    ],
    "Value": [
        "RQ3 - Interpretability of the XGBoost cost overrun model",
        "Which project characteristics drive the model's predictions, and are "
        "they consistent with the megaproject literature?",
        f"{n} megaprojects (Meets_500M_2025 = Yes)",
        "log(Orig_USD_PPP_2025_M), Sector_Std, Source",
        "ln(1 + Cost_Overrun_Pct / 100)",
        "SHAP TreeExplainer (exact Shapley values for tree ensembles)",
        "Log cost-ratio units; a contribution s multiplies the predicted "
        "cost ratio by exp(s)",
        f"{N_SPLITS}-fold cross-validation repeated {N_REPEATS} times; each "
        "project explained only by models that did not train on it",
        f"{N_SPLITS * N_REPEATS}",
        "One-hot encoder fitted on the full predictor matrix so SHAP columns "
        "are stable across folds; no target information is used",
        "One-hot contributions summed to the parent variable before taking "
        "absolute values",
        f"Reproduced RQ2 out-of-fold MAE of {RQ2_XGB_MAE} pp "
        f"(obtained {mae_here:.3f} pp)",
        f"Model refitted without Source; MAE {mae_s:.3f} pp",
        "Linear Regression coefficients on the same encoded design matrix",
        str(RANDOM_STATE),
    ],
})

wb = openpyxl.Workbook()
write_sheet(wb, "RQ3_Importance_Variable", importance, first=True)
write_sheet(wb, "RQ3_Importance_Encoded", importance_enc)
write_sheet(wb, "RQ3_ProjectSHAP", project_shap)
write_sheet(wb, "RQ3_LocalCases", local_cases)
write_sheet(wb, "RQ3_LR_Coefficients", lr_coefs)
write_sheet(wb, "RQ3_Sensitivity_NoSource", sensitivity)
ws = write_sheet(wb, "Config", config_rows)
ws.column_dimensions["A"].width = 30
ws.column_dimensions["B"].width = 110
for row in ws.iter_rows(min_row=2):
    row[0].font = BOLD
    row[0].alignment = Alignment(vertical="top", wrap_text=True)
    row[1].alignment = Alignment(vertical="top", wrap_text=True)

wb.save(OUTPUT_XLSX)

print(f"Saved workbook: {OUTPUT_XLSX}")
print("Sheets:", wb.sheetnames)
print(f"Saved figures to: {FIG_DIR}/")
print("\nFigures produced:")
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith(".png"):
        print("  " + f)

In [ ]:
# ------------------------------------------------------------
# 15. Download everything (Colab)
# ------------------------------------------------------------

import shutil
from google.colab import files

shutil.make_archive("RQ3_Figures", "zip", FIG_DIR)

files.download(OUTPUT_XLSX)
files.download("RQ3_Figures.zip")

## Notes for the write-up

**Attribution scale.** Do not report SHAP values as percentage points. They are in
`ln(1 + overrun/100)` units. State this once in the methodology, then either report the
raw values or the multiplicative translation from the importance table.

**Out-of-fold attribution.** Most published SHAP applications explain a single model
fitted on all the data. Explaining held-out predictions across repeated cross-validation
is more conservative and directly comparable to the RQ2 performance estimates. This is a
methodological choice worth defending in a sentence, and worth citing as a strength.

**Stability.** The `SD_across_repeats` column is your evidence that the importance
ranking is not an artefact of one particular fold split. With N=107 this matters; report
it rather than the point estimate alone.

**The link back to RQ2.** RQ2 found neither model beat the on-budget baseline on the
±10pp criterion, and that both systematically under-predict large overruns. RQ3 should
explain *why*: if the model has only size, sector and provenance to work with, it has no
access to the mechanisms the literature identifies as driving severe overrun — scope
change, optimism bias, governance quality, design maturity. A low-importance,
low-accuracy result is a coherent finding, not a failed experiment, provided you argue it
that way.